<a href="https://colab.research.google.com/github/Page0526/survival-prediction/blob/main/multimodal_fusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Download Data and Install Libs

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from google.colab import userdata
hf_key = userdata.get('huggingface')

In [3]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [21]:
page0526_tcga_5_subset_path = kagglehub.dataset_download('page0526/tcga-5-subset')
print('Data source import complete.')

Data source import complete.


In [6]:
!pip install lifelines

In [9]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? [y/N]: y
Token is valid (permission: fineGrained).
The token `uni` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate whe

In [10]:
dataset_name = "TCGA"
project_name = "TCGA-BLCA"
local_dir = "./drive/MyDrive/UNI2-h_features"

file_path = f"{dataset_name}/{project_name}.tar.gz"

!hf download MahmoodLab/UNI2-h-features \
    "$file_path" \
    --repo-type dataset \
    --local-dir "$local_dir"

TCGA/TCGA-BLCA.tar.gz: 100% 31.9G/31.9G [05:15<00:00, 101MB/s] 
✓ Downloaded
  path: drive/MyDrive/UNI2-h_features/TCGA/TCGA-BLCA.tar.gz


In [14]:
!mkdir -p /content/UNI2-h_features/TCGA

In [15]:
!tar -xzf /content/drive/MyDrive/UNI2-h_features/TCGA/TCGA-BLCA.tar.gz -C /content/UNI2-h_features/TCGA/

In [17]:
import h5py
h5_file = '/content/UNI2-h_features/TCGA/TCGA-2F-A9KO-01Z-00-DX1.195576CF-B739-4BD9-B15B-4A70AE287D3E.h5'
with h5py.File(h5_file,'r') as file:
    features = file['features'][:] # 1 x num_patches x 1536
    coords = file['coords'][:] # 1 x num_patches x 2

# Exploratory Data Analysis

# Finetuning Foundation Model for Multimodal Fusion Between WSI and Genomic Data

In [29]:
import os, copy, warnings, h5py
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)


# ─────────────────────────────────────────────────────────────
# 1. CONFIG
# ─────────────────────────────────────────────────────────────

# Feature dimensions per foundation model
_MODEL_DIMS = {
    "conch": 512,    # CONCH / TITAN pre-extracted features
    "uni":   1536,   # UNI ViT-L/16
    "plip":  512,    # PLIP (if you have those)
    "resnet": 1024,  # PORPOISE ResNet-50 baseline
}

class Config:
    DATA_DIR    = f"page0526_tcga_5_subset_path"          # genomic CSVs
    WSI_H5      = "/content/UNI2-h_features/TCGA"  # single HDF5, keys = slide_id

    MODEL       = "uni"

    @property
    def WSI_FEAT_DIM(self):
        return _MODEL_DIMS[self.MODEL]

    # ── Data columns ─────────────────────────────────────────
    # CANCER_TYPES    = ["BLCA", "BRCA", "GBMLGG", "LUAD", "UCEC"]
    CANCER_TYPES    = ["BLCA"]
    SLIDE_ID_COL    = "slide_id"
    CLINICAL_COLS   = ["is_female", "age"]
    SURVIVAL_COL    = "survival_months"
    EVENT_COL       = "censorship"
    TRAIN_COL       = "train"
    CNV_SUFFIX      = "_cnv"
    RNASEQ_SUFFIX   = "_rnaseq"

    # ── Genomic feature selection ─────────────────────────────
    VARIANCE_THRESHOLD = 0.01
    TOP_K_CNV          = 30
    TOP_K_RNASEQ       = 100
    # Number of genomic tokens (pathway-like groups fed to co-attention)
    N_GENOMIC_TOKENS   = 6

    # ── MCAT model ───────────────────────────────────────────
    HIDDEN_DIM  = 256           # shared dim throughout
    N_HEADS     = 4
    DROPOUT     = 0.25
    N_CLASSES   = 4             # survival time bins

    # ── SNN genomic encoder ──────────────────────────────────
    SNN_HIDDEN  = [256, 256, 256, 256]   # SNN layer sizes
    SNN_DROP_1  = 0.3
    SNN_DROP    = 0.6

    # ── Training ─────────────────────────────────────────────
    EPOCHS      = 100
    LR          = 2e-4
    WEIGHT_DECAY= 1e-3
    GRAD_ACCUM  = 8      # effective batch size = 8 slides
    MAX_PATCHES = 2048   # cap per slide (T4: 15 GB VRAM safety)
    PATIENCE    = 20
    LR_PATIENCE = 8
    LR_FACTOR   = 0.5
    LR_MIN      = 1e-6
    N_FOLDS     = 5
    DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

    SEED        = 42
    SAVE_DIR    = "./checkpoints"
    LOG_EVERY   = 5

# ─────────────────────────────────────────────────────────────
# 4. SNN GENOMIC TOKENIZER
# ─────────────────────────────────────────────────────────────

class SNN_Block(nn.Module):
    """Linear → SELU → AlphaDropout  (self-normalizing block)."""
    def __init__(self, d_in, d_out, dropout=0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_out), nn.SELU(), nn.AlphaDropout(p=dropout))
    def forward(self, x): return self.net(x)


def _init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.trunc_normal_(m.weight, std=1./np.sqrt(m.weight.size(1)))
        if m.bias is not None: nn.init.zeros_(m.bias)


class GenomicTokenizer(nn.Module):
    """
    Flat genomic vector  →  G genomic tokens  (MCAT pathway tokenisation).

    Step 1: SNN encodes the full feature vector into a rich embedding.
    Step 2: Split embedding into G equal groups; project each to hidden_dim.
            Each group acts as one "pathway token" for co-attention.

    Using groups avoids needing an external pathway database while still
    giving the co-attention G distinct genomic perspectives to attend over.
    """
    def __init__(self, input_dim, n_tokens, hidden_dim,
                 snn_hidden, drop_first, dropout):
        super().__init__()
        dims   = [input_dim] + snn_hidden
        blocks = [SNN_Block(dims[0], dims[1], dropout=drop_first)]
        for i in range(1, len(dims)-1):
            blocks.append(SNN_Block(dims[i], dims[i+1], dropout=dropout))
        self.snn = nn.Sequential(*blocks)
        snn_out  = snn_hidden[-1]

        # Pad snn_out so it divides evenly into n_tokens groups
        self.n_tokens   = n_tokens
        self.group_size = (snn_out + n_tokens - 1) // n_tokens
        self.pad        = self.group_size * n_tokens - snn_out

        self.projs = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.group_size, hidden_dim),
                nn.LayerNorm(hidden_dim),
            )
            for _ in range(n_tokens)
        ])
        self.apply(_init_weights)

    def forward(self, x):          # (B, F) → (B, G, D)
        h = self.snn(x)
        if self.pad: h = F.pad(h, (0, self.pad))
        chunks = h.chunk(self.n_tokens, dim=-1)
        return torch.stack([self.projs[i](chunks[i])
                            for i in range(self.n_tokens)], dim=1)


# ─────────────────────────────────────────────────────────────
# 5. WSI PATCH PROJECTOR
# ─────────────────────────────────────────────────────────────

class PatchProjector(nn.Module):
    """
    Project frozen foundation-model patch embeddings → hidden_dim.
    Simple: no parameters for the FM itself (features are pre-extracted).
    """
    def __init__(self, feat_dim, hidden_dim, dropout=0.1):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
    def forward(self, x): return self.proj(x)    # (N, D)


# ─────────────────────────────────────────────────────────────
# 6. MCAT CO-ATTENTION  (Chen et al., NeurIPS 2021)
# ─────────────────────────────────────────────────────────────

class CoAttentionLayer(nn.Module):
    """
    One MCAT co-attention layer.

    Two cross-attention streams run in parallel:
      g→p : genomic tokens query WSI patches
            "which tissue regions match this molecular profile?"
      p→g : WSI patches query genomic tokens
            "which gene groups explain these morphological patterns?"

    Each stream is followed by self-attention + FFN (standard Transformer block).
    Outputs are mean-pooled into fixed-size vectors.
    """
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        kw = dict(embed_dim=d, num_heads=n_heads,
                  dropout=dropout, batch_first=True)
        self.x_g2p = nn.MultiheadAttention(**kw)   # genomic → WSI
        self.x_p2g = nn.MultiheadAttention(**kw)   # WSI → genomic
        self.s_g   = nn.MultiheadAttention(**kw)   # genomic self-attn
        self.s_p   = nn.MultiheadAttention(**kw)   # WSI self-attn

        self.n_g1  = nn.LayerNorm(d); self.n_g2 = nn.LayerNorm(d)
        self.n_p1  = nn.LayerNorm(d); self.n_p2 = nn.LayerNorm(d)

        ff = lambda: nn.Sequential(
            nn.Linear(d, d*2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d*2, d))
        self.ff_g = ff(); self.ff_p = ff()

    def forward(self, g, p):
        """g: (B,G,D)  p: (B,N,D)  →  g_out: (B,D)  p_out: (B,D)"""
        # Cross-attention
        g = self.n_g1(g + self.x_g2p(g, p, p)[0])
        p = self.n_p1(p + self.x_p2g(p, g, g)[0])
        # Self-attention + FFN
        g = self.n_g2(g + self.s_g(g, g, g)[0]); g = g + self.ff_g(g)
        p = self.n_p2(p + self.s_p(p, p, p)[0]); p = p + self.ff_p(p)
        return g.mean(1), p.mean(1)                # (B,D), (B,D)


# ─────────────────────────────────────────────────────────────
# 7. FULL MCAT SURVIVAL MODEL
# ─────────────────────────────────────────────────────────────

class MCATSurvival(nn.Module):
    """
    Input:
      patches  (N, wsi_feat_dim)  — pre-extracted UNI/CONCH patch features
      genomic  (1, genomic_dim)   — scaled genomic features

    Pipeline:
      genomic → SNN → 6 tokens  (1, 6, D)
      patches → proj             (1, N, D)
                  ↕  co-attention
      g_agg (1,D)  p_agg (1,D)
           → concat → head → hazards (1, n_classes)

    Output: (hazards, S, Y_hat)  — same interface as genomic-only SNN.
    """
    def __init__(self, genomic_dim: int, cfg: Config):
        super().__init__()
        D = cfg.HIDDEN_DIM

        self.genomic_tok = GenomicTokenizer(
            input_dim  = genomic_dim,
            n_tokens   = cfg.N_GENOMIC_TOKENS,
            hidden_dim = D,
            snn_hidden = cfg.SNN_HIDDEN,
            drop_first = cfg.SNN_DROP_1,
            dropout    = cfg.SNN_DROP,
        )
        self.patch_proj = PatchProjector(cfg.WSI_FEAT_DIM, D, dropout=0.1)
        self.coattn     = CoAttentionLayer(D, cfg.N_HEADS, cfg.DROPOUT)
        self.head       = nn.Sequential(
            nn.Linear(D * 2, D), nn.ReLU(), nn.Dropout(cfg.DROPOUT),
            nn.Linear(D, cfg.N_CLASSES),
        )
        self.apply(_init_weights)
        self.cfg = cfg

    def forward(self, patches: torch.Tensor, genomic: torch.Tensor):
        """
        patches : (N, feat_dim)       variable N per slide
        genomic : (1, genomic_dim)
        """
        g_tok  = self.genomic_tok(genomic)           # (1, G, D)
        p_tok  = self.patch_proj(patches).unsqueeze(0)  # (1, N, D)

        g_agg, p_agg = self.coattn(g_tok, p_tok)    # (1,D), (1,D)

        logits  = self.head(torch.cat([g_agg, p_agg], dim=-1))  # (1, C)
        hazards = torch.sigmoid(logits)
        S       = torch.cumprod(1 - hazards, dim=1)
        Y_hat   = logits.argmax(dim=1, keepdim=True)
        return hazards, S, Y_hat

    def relocate(self):
        self.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

    @torch.no_grad()
    def attention_heatmap(self, patches: torch.Tensor,
                          genomic: torch.Tensor) -> np.ndarray:
        """
        Returns (G, N) attention weights  (genomic tokens → patch positions).
        Useful to visualise which tissue regions each 'pathway' attends to.
        Average over heads for interpretability.
        """
        self.eval()
        g_tok = self.genomic_tok(genomic)
        p_tok = self.patch_proj(patches).unsqueeze(0)
        _, w  = self.coattn.x_g2p(
            g_tok, p_tok, p_tok,
            need_weights=True, average_attn_weights=True)  # (1,G,N)
        return w.squeeze(0).cpu().numpy()                   # (G, N)


# ─────────────────────────────────────────────────────────────
# 8. SURVIVAL LOSS + BINNING
# ─────────────────────────────────────────────────────────────

def nll_loss(hazards, S, Y, c):
    """Discrete-time NLL.  c: 1=censored, 0=event."""
    S_pad  = torch.cat([torch.ones_like(S[:,:1]), S], dim=1)
    idx    = Y.view(-1,1)
    h_t    = torch.gather(hazards, 1, idx).squeeze(1)
    s_prev = torch.gather(S_pad,   1, idx).squeeze(1)
    s_t    = torch.gather(S,       1, idx).squeeze(1)
    return ((1-c)*(-torch.log(h_t+1e-7) - torch.log(s_prev+1e-7))
            + c*(-torch.log(s_t+1e-7))).mean()


def bin_times(t, n, eps=1e-6, cuts=None):
    if cuts is None:
        cuts = np.quantile(t, np.linspace(0,1,n+1))
        cuts[0] -= eps; cuts[-1] += eps
    return np.clip(np.digitize(t, cuts[1:-1]), 0, n-1).astype(int), cuts


# ─────────────────────────────────────────────────────────────
# 9. GENOMIC FEATURE SELECTION
# ─────────────────────────────────────────────────────────────

def logrank_select(X, y, k, t_col, e_col):
    print(f"    log-rank {X.shape[1]} → {k}", end=" … ", flush=True)
    T, E = y[t_col].values, y[e_col].values
    pv = {}
    for c in X.columns:
        v = X[c].values; m = np.median(v)
        lo, hi = v<=m, v>m
        pv[c] = logrank_test(T[lo],T[hi],E[lo],E[hi]).p_value \
                if lo.sum()>=2 and hi.sum()>=2 else 1.0
    sel = sorted(pv, key=pv.get)[:k]
    print(f"done (best p={pv[sel[0]]:.1e})")
    return sel


def select_features(train_df, cnv_cols, rna_cols, cfg):
    y = train_df[[cfg.SURVIVAL_COL, cfg.EVENT_COL]]
    def vf(cols):
        vt = VarianceThreshold(cfg.VARIANCE_THRESHOLD)
        vt.fit(train_df[cols])
        return [c for c,ok in zip(cols, vt.get_support()) if ok]
    cnv_v = vf(cnv_cols) if cnv_cols else []
    rna_v = vf(rna_cols)
    sel_cnv = logrank_select(train_df[cnv_v], y,
                             min(cfg.TOP_K_CNV, len(cnv_v)),
                             cfg.SURVIVAL_COL, cfg.EVENT_COL) if cnv_v else []
    sel_rna = logrank_select(train_df[rna_v], y,
                             min(cfg.TOP_K_RNASEQ, len(rna_v)),
                             cfg.SURVIVAL_COL, cfg.EVENT_COL)
    return sel_cnv, sel_rna


# ─────────────────────────────────────────────────────────────
# 10. DATASET — lazy HDF5 slide loading
# ─────────────────────────────────────────────────────────────

class MultimodalDataset(Dataset):
    """
    Each sample = one patient.
    Patches loaded lazily from HDF5 by slide_id key.
    HDF5 handle is opened once per worker; no multiprocessing fork issues
    as long as num_workers=0 (required on Colab).
    """
    def __init__(self, df, feat_cols, bins, cfg):
        self.df        = df.reset_index(drop=True)
        self.feat_cols = feat_cols
        self.bins      = bins
        self.cfg       = cfg
        self._h5       = None           # opened lazily in __getitem__

        # Filter to slides present in HDF5
        present = self._slide_keys()
        mask    = self.df[cfg.SLIDE_ID_COL].isin(present)
        n_miss  = (~mask).sum()
        if n_miss:
            print(f"  [dataset] {n_miss} slides not in HDF5 → excluded")
        self.df   = self.df[mask].reset_index(drop=True)
        self.bins = self.bins[mask.values]
        print(f"  [dataset] {len(self.df)} usable slides")

    def _slide_keys(self):
        if not Path(self.cfg.WSI_H5).exists():
            print(f"  [warn] HDF5 not found: {self.cfg.WSI_H5}")
            return set()
        with h5py.File(self.cfg.WSI_H5, "r") as hf:
            return set(hf.keys())

    def _open(self):
        """Open HDF5 on first access (safe with num_workers=0)."""
        if self._h5 is None:
            self._h5 = h5py.File(self.cfg.WSI_H5, "r")

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        self._open()
        row     = self.df.iloc[idx]
        sid     = str(row[self.cfg.SLIDE_ID_COL])

        # Load patches from HDF5
        patches = torch.tensor(self._h5[sid][:], dtype=torch.float32)

        # Random subsample if slide is very large
        if len(patches) > self.cfg.MAX_PATCHES:
            sel     = torch.randperm(len(patches))[:self.cfg.MAX_PATCHES]
            patches = patches[sel]

        genomic = torch.tensor(
            row[self.feat_cols].values.astype(np.float64),
            dtype=torch.float32).unsqueeze(0)                # (1, F)

        Y = torch.tensor(int(self.bins[idx]), dtype=torch.long)
        c = torch.tensor(1.0 - float(row[self.cfg.EVENT_COL]),
                         dtype=torch.float32)
        return (patches, genomic, Y, c,
                float(row[self.cfg.SURVIVAL_COL]),
                bool(row[self.cfg.EVENT_COL]))

    def __del__(self):
        if self._h5 is not None:
            try: self._h5.close()
            except: pass


def _collate(batch):
    return ([b[0] for b in batch],
            torch.cat([b[1] for b in batch]),
            torch.stack([b[2] for b in batch]),
            torch.stack([b[3] for b in batch]),
            np.array([b[4] for b in batch]),
            np.array([b[5] for b in batch]))


# ─────────────────────────────────────────────────────────────
# 11. DATA LOADING
# ─────────────────────────────────────────────────────────────

def load_raw(cfg):
    frames = []
    # for ct in cfg.CANCER_TYPES:
    #     p = Path(cfg.DATA_DIR) / f"tcga_{ct}_all_clean.csv"
    #     if p.exists():
    #         df = pd.read_csv(p); df["cancer_type"] = ct; frames.append(df)
    # if not frames:
    #     mp = Path(cfg.DATA_DIR) / "merged.csv"
    #     if not mp.exists():
    #         raise FileNotFoundError(cfg.DATA_DIR)
    #     frames.append(pd.read_csv(mp))
    # df = pd.concat(frames, ignore_index=True)
    p = "/content/genes/tcga_blca_all_clean.csv"
    df = pd.read_csv(p)

    df = df[pd.to_numeric(df[cfg.SURVIVAL_COL],errors="coerce").notna()
            ].reset_index(drop=True)
    cnv  = [c for c in df.columns if c.endswith(cfg.CNV_SUFFIX)]
    rna  = [c for c in df.columns if c.endswith(cfg.RNASEQ_SUFFIX)]
    for col in cfg.CLINICAL_COLS+cnv+rna+[cfg.SURVIVAL_COL,cfg.EVENT_COL]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=cfg.CLINICAL_COLS+cnv+rna+
                   [cfg.SURVIVAL_COL,cfg.EVENT_COL,cfg.SLIDE_ID_COL])
    df = df[df[cfg.SURVIVAL_COL]>0].reset_index(drop=True)
    print(f"Samples: {len(df):,}  CNV={len(cnv)}  RNA={len(rna)}")
    return df, cnv, rna


def make_loaders(tr_df, va_df, cnv, rna, cfg):
    print("Feature selection …")
    sel_cnv, sel_rna = select_features(tr_df, cnv, rna, cfg)
    feat_cols = cfg.CLINICAL_COLS + sel_cnv + sel_rna
    print(f"  Features: {len(feat_cols)}  "
          f"(clin={len(cfg.CLINICAL_COLS)} "
          f"cnv={len(sel_cnv)} rna={len(sel_rna)})")

    sc = StandardScaler()
    tr_df = tr_df.copy(); va_df = va_df.copy()
    tr_df[feat_cols] = sc.fit_transform(tr_df[feat_cols])
    va_df[feat_cols] = sc.transform(va_df[feat_cols])

    tr_bins, cuts = bin_times(tr_df[cfg.SURVIVAL_COL].values, cfg.N_CLASSES)
    va_bins, _    = bin_times(va_df[cfg.SURVIVAL_COL].values,
                              cfg.N_CLASSES, cuts=cuts)

    tr_ds = MultimodalDataset(tr_df, feat_cols, tr_bins, cfg)
    va_ds = MultimodalDataset(va_df, feat_cols, va_bins, cfg)

    kw = dict(collate_fn=_collate, num_workers=0, pin_memory=False)
    return (DataLoader(tr_ds, batch_size=1, shuffle=True, **kw),
            DataLoader(va_ds, batch_size=1, shuffle=False, **kw),
            dict(feat_cols=feat_cols, scaler=sc, cuts=cuts))


# ─────────────────────────────────────────────────────────────
# 12. EVALUATION
# ─────────────────────────────────────────────────────────────

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    risks, Ts, Es = [], [], []
    for patches_l, genomic, Y, c, T, E in loader:
        haz, *_ = model(patches_l[0].to(device), genomic.to(device))
        risks.append(haz.mean().item())
        Ts.append(T[0]); Es.append(E[0])
    return concordance_index(np.array(Ts), -np.array(risks),
                             np.array(Es, dtype=bool))


# ─────────────────────────────────────────────────────────────
# 13. TRAINING
# ─────────────────────────────────────────────────────────────

def _train_fold(tr_loader, va_loader, meta, cfg, verbose=True):
    dev   = cfg.DEVICE
    n_dim = len(meta["feat_cols"])
    model = MCATSurvival(n_dim, cfg); model.relocate()

    n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if verbose:
        print(f"  Model: {cfg.MODEL.upper()} feats={cfg.WSI_FEAT_DIM}  "
              f"genomic={n_dim}  tokens={cfg.N_GENOMIC_TOKENS}  "
              f"hidden={cfg.HIDDEN_DIM}  params={n_p:,}")

    opt  = optim.AdamW(model.parameters(), lr=cfg.LR,
                       weight_decay=cfg.WEIGHT_DECAY)
    sched= optim.lr_scheduler.ReduceLROnPlateau(
        opt, "max", cfg.LR_FACTOR, cfg.LR_PATIENCE, min_lr=cfg.LR_MIN)

    best_ci, pat, best_w = 0.0, 0, None
    hist = {"loss":[], "tr_ci":[], "va_ci":[]}

    for ep in range(1, cfg.EPOCHS+1):
        model.train(); ep_loss = 0.0; opt.zero_grad()

        for step, (patches_l, genomic, Y, c, _, _) in enumerate(tr_loader):
            haz, S, _ = model(patches_l[0].to(dev), genomic.to(dev))
            loss = nll_loss(haz, S, Y.to(dev), c.to(dev)) / cfg.GRAD_ACCUM
            loss.backward(); ep_loss += loss.item() * cfg.GRAD_ACCUM

            if (step+1) % cfg.GRAD_ACCUM == 0:
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                opt.step(); opt.zero_grad()

        # flush remaining
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step(); opt.zero_grad()

        avg = ep_loss / len(tr_loader)
        tr  = evaluate(model, tr_loader, dev)
        va  = evaluate(model, va_loader, dev)
        sched.step(va)
        lr  = opt.param_groups[0]["lr"]

        hist["loss"].append(avg); hist["tr_ci"].append(tr)
        hist["va_ci"].append(va)

        if verbose and (ep % cfg.LOG_EVERY == 0 or ep == 1):
            print(f"  Ep {ep:03d}/{cfg.EPOCHS}  loss={avg:.4f}  "
                  f"tr_ci={tr:.4f}  va_ci={va:.4f}  lr={lr:.1e}")

        if va > best_ci:
            best_ci = va; pat = 0
            best_w  = copy.deepcopy(model.state_dict())
        else:
            pat += 1
            if pat >= cfg.PATIENCE:
                if verbose: print(f"  Early stop ep {ep}")
                break

    if best_w: model.load_state_dict(best_w)
    return model, hist, best_ci


# ─────────────────────────────────────────────────────────────
# 14. CROSS-VALIDATION
# ─────────────────────────────────────────────────────────────

def cross_validate(cfg: Config, n_folds: int = None):
    n_folds    = n_folds or cfg.N_FOLDS
    cfg.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(cfg.SEED); np.random.seed(cfg.SEED)
    os.makedirs(cfg.SAVE_DIR, exist_ok=True)

    print("=" * 62)
    print(f"MCAT Survival  model={cfg.MODEL.upper()}  "
          f"feat_dim={cfg.WSI_FEAT_DIM}  device={cfg.DEVICE}")
    print("=" * 62)
    df, cnv, rna = load_raw(cfg)
    skf  = StratifiedKFold(n_folds, shuffle=True, random_state=cfg.SEED)
    evts = df[cfg.EVENT_COL].astype(int).values
    scores = []

    for fold, (tr_i, va_i) in enumerate(skf.split(df, evts)):
        print(f"\n─── Fold {fold+1}/{n_folds}  "
              f"train={len(tr_i)} val={len(va_i)} ───")
        tr_df = df.iloc[tr_i].copy().reset_index(drop=True)
        va_df = df.iloc[va_i].copy().reset_index(drop=True)
        tr_l, va_l, meta = make_loaders(tr_df, va_df, cnv, rna, cfg)
        _, _, ci = _train_fold(tr_l, va_l, meta, copy.deepcopy(cfg))
        scores.append(ci)
        print(f"  Fold {fold+1} best C-index: {ci:.4f}")

    print(f"\n{'='*62}")
    print(f"CV  ({n_folds} folds)  mean={np.mean(scores):.4f}  "
          f"std={np.std(scores):.4f}")
    for i,s in enumerate(scores,1): print(f"  Fold {i}: {s:.4f}")
    return scores


# ─────────────────────────────────────────────────────────────
# 15. SINGLE TRAIN RUN + SAVE
# ─────────────────────────────────────────────────────────────

def train(cfg: Config):
    cfg.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(cfg.SEED); np.random.seed(cfg.SEED)
    os.makedirs(cfg.SAVE_DIR, exist_ok=True)
    df, cnv, rna = load_raw(cfg)

    tc   = pd.to_numeric(df[cfg.TRAIN_COL], errors="coerce")
    mask = tc.fillna(0).astype(int) == 1
    tr_l, va_l, meta = make_loaders(
        df[mask].copy(), df[~mask].copy(), cnv, rna, cfg)

    model, _, ci = _train_fold(tr_l, va_l, meta, cfg)
    torch.save({"model": model.state_dict(), "meta": meta,
                "config": cfg.__dict__, "val_ci": ci},
               Path(cfg.SAVE_DIR) / "best_mcat.pt")
    print(f"Best val C-index: {ci:.4f}  saved → {cfg.SAVE_DIR}/best_mcat.pt")
    return model

In [30]:
cfg = Config()
cfg.MODEL    = "uni"
cfg.DATA_DIR = "/content/genes/"
cfg.WSI_H5   = "/content/UNI2-h_features/TCGA"
cross_validate(cfg)

MCAT Survival  model=UNI  feat_dim=1536  device=cpu
Samples: 437  CNV=1656  RNA=18369

─── Fold 1/5  train=349 val=88 ───
Feature selection …
    log-rank 1656 → 30 … done (best p=1.1e-03)
    log-rank 18351 → 100 … done (best p=1.2e-09)
  Features: 132  (clin=2 cnv=30 rna=100)


IsADirectoryError: [Errno 21] Unable to synchronously open file (file read failed: time = Sun May 10 11:19:40 2026
, filename = '/content/UNI2-h_features/TCGA', file descriptor = 48, errno = 21, error message = 'Is a directory', buf = 0x7ffe20fbe338, total read size = 8, bytes this sub-read = 8, offset = 0)